In [1]:
%pip install yfinance

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.2/130.2 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 9.4 MB/s eta 0:00:00a 0:00:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.4/139.4 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 429.2/429.2 kB 12.0 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 7.2 MB/s eta 0:00:00
  Created wheel for multitasking: filename=multitasking-0.0.12-py3-none-any.whl size=15635 sha256=8bd0c3fa6d0f3f6891bdc3bc7149f920c9375163adcb6f2df223be24191e1163
  Stored in directory: /Users/bcslingluff/Library/Caches/pip/wheels/42/d6/84/bf57a755f4569494cd00de4bb46ef064874823f4d19c82e960
Successfully built multitasking

[notice] A new release of pip is available: 24.1.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may nee

In [2]:
import pandas as pd
import yfinance as yf
import numpy as np

In [4]:
import os

current_directory = os.getcwd()
print(current_directory)

/Users/bcslingluff/Desktop/DS4002/Project-2-Bitcoin/SCRIPTS


In [5]:
# Load Bitcoin Data
btc = pd.read_csv('../DATA/btc-usd-max.csv')
btc['date'] = pd.to_datetime(btc['snapped_at']).dt.date
btc = btc.rename(columns={'price': 'btc_price', 'market_cap': 'market_cap_btc', 'total_volume': 'trading_vol_btc'})
btc = btc[['date', 'btc_price', 'market_cap_btc', 'trading_vol_btc']]
btc['date'] = pd.to_datetime(btc['date'])

# Load 5-Year Breakeven Inflation Data
t5yie = pd.read_csv('../DATA/T5YIE.csv')
t5yie = t5yie.rename(columns={'observation_date': 'date', 'T5YIE': 'breakeven_5y'})
t5yie['date'] = pd.to_datetime(t5yie['date'])
t5yie['breakeven_5y'] = pd.to_numeric(t5yie['breakeven_5y'], errors='coerce')

btc.head(5) # Let Jupyter render the dataframe so you can inspect it

,date,btc_price,market_cap_btc,trading_vol_btc
0,2013-04-28,135.30,1.500518e+09,0.0
1,2013-04-29,141.96,1.575032e+09,0.0
2,2013-04-30,135.30,1.501657e+09,0.0
3,2013-05-01,117.00,1.298952e+09,0.0
4,2013-05-02,103.43,1.148668e+09,0.0


In [6]:
start_date = "2013-04-28"
tickers = ["^GSPC", "DX-Y.NYB"]

print("Downloading yfinance data...")
yf_data = yf.download(tickers, start=start_date)['Close']
yf_data = yf_data.reset_index()
yf_data = yf_data.rename(columns={'Date': 'date', '^GSPC': 'sp500_close', 'DX-Y.NYB': 'dxy_close'})
yf_data['date'] = pd.to_datetime(yf_data['date'])

yf_data.head(5)

[*********************100%***********************]  2 of 2 completed


Ticker,date,dxy_close,sp500_close
0,2013-04-29,82.150002,1593.609985
1,2013-04-30,81.730003,1597.569946
2,2013-05-01,81.639999,1582.699951
3,2013-05-02,82.209999,1597.589966
4,2013-05-03,82.099998,1614.420044


In [8]:
daily_df = pd.merge(btc, t5yie, on='date', how='left')
daily_df = pd.merge(daily_df, yf_data, on='date', how='left')

# Forward fill missing weekend data
daily_df[['breakeven_5y', 'sp500_close', 'dxy_close']] = daily_df[['breakeven_5y', 'sp500_close', 'dxy_close']].ffill()

daily_df['month'] = daily_df['date'].dt.to_period('M')
btc_monthly = daily_df.groupby('month')['btc_price'].mean().reset_index()
btc_monthly = btc_monthly.rename(columns={'btc_price': 'btc_monthly_avg'})

daily_df[['breakeven_5y', 'sp500_close', 'dxy_close']] = daily_df[['breakeven_5y', 'sp500_close', 'dxy_close']].bfill()
# Check for any remaining missing values
daily_df.isnull().sum()

date               0
btc_price          0
market_cap_btc     0
trading_vol_btc    0
breakeven_5y       0
dxy_close          0
sp500_close        0
month              0
dtype: int64

In [11]:
cpi = pd.read_csv('../DATA/CPIAUCSL.csv')
cpi = cpi.rename(columns={'observation_date': 'date', 'CPIAUCSL': 'cpi_raw'})
cpi['date'] = pd.to_datetime(cpi['date'])
cpi['month'] = cpi['date'].dt.to_period('M')
cpi['cpi_yoy_pct'] = cpi['cpi_raw'].pct_change(12, fill_method=None) * 100
cpi = cpi[['month', 'cpi_raw', 'cpi_yoy_pct']]

# Create Master DataFrames
monthly_master = pd.merge(cpi, btc_monthly, on='month', how='inner')
master_df = pd.merge(daily_df, cpi, on='month', how='left')
master_df = pd.merge(master_df, btc_monthly, on='month', how='left')

master_df['month'] = master_df['month'].astype(str)
monthly_master['month'] = monthly_master['month'].astype(str)

master_df.to_csv('master_dataset_daily.csv', index=False)
monthly_master.to_csv('master_dataset_monthly.csv', index=False)
print("Data fully merged and saved!")

master_df.head()


Data fully merged and saved!


,date,btc_price,market_cap_btc,trading_vol_btc,breakeven_5y,dxy_close,sp500_close,month,cpi_raw,cpi_yoy_pct,btc_monthly_avg
0,2013-04-28,135.30,1.500518e+09,0.0,2.10,82.150002,1593.609985,2013-04,231.797,NaN,137.520000
1,2013-04-29,141.96,1.575032e+09,0.0,2.10,82.150002,1593.609985,2013-04,231.797,NaN,137.520000
2,2013-04-30,135.30,1.501657e+09,0.0,2.06,81.730003,1597.569946,2013-04,231.797,NaN,137.520000
3,2013-05-01,117.00,1.298952e+09,0.0,1.98,81.639999,1582.699951,2013-05,231.893,NaN,119.423968
4,2013-05-02,103.43,1.148668e+09,0.0,1.99,82.209999,1597.589966,2013-05,231.893,NaN,119.423968


In [ ]:
master_df.tail()

,date,btc_price,market_cap_btc,trading_vol_btc,breakeven_5y,dxy_close,sp500_close,month,cpi_raw,cpi_yoy_pct,btc_monthly_avg
2484,2020-02-17,9943.673603,1.812215e+11,4.236491e+10,1.64,99.120003,3380.159912,2020-02,259.250,2.341317,9661.306805
3930,2024-02-02,43069.043421,8.445852e+11,2.244368e+10,2.19,103.919998,4958.609863,2024-02,310.967,3.157074,49232.519618
4376,2025-04-23,93576.165886,1.858588e+12,5.384199e+10,2.33,99.839996,5375.859863,2025-04,320.302,2.325388,86068.125203
3555,2023-01-23,22736.661429,4.380604e+11,3.412436e+10,2.28,102.139999,4019.810059,2023-01,300.420,6.327179,20044.788880
4120,2024-08-10,60912.588533,1.198301e+12,3.380421e+10,1.96,103.139999,5344.160156,2024-08,314.062,2.607144,60107.968531
